# 02 Data Preparation\n
Normalize datasets into instruction, context, answer JSONL files.

In [ ]:
import json
import os
from pathlib import Path
from datasets import load_dataset
from sklearn.model_selection import train_test_split
import numpy as np

print("Imports successful!")

In [ ]:
print("Loading FinQA dataset...")
dataset = load_dataset('ibm/finqa', split='train')

print(f"Total samples: {len(dataset)}")

# Convert to instruction-following format
rows = []
for idx, x in enumerate(dataset):
    question = x.get('question', '')
    answer = x.get('answer', '')
    pre_text = x.get('pre_text', '')
    post_text = x.get('post_text', '')
    
    # Combine context
    context = f"{pre_text}\n{post_text}".strip()
    
    row = {
        'instruction': question,
        'context': context,
        'output': str(answer)
    }
    rows.append(row)

print(f"✓ Converted {len(rows)} samples")
print(f"\nSample format:")
print(json.dumps(rows[0], indent=2)[:300] + "...")

In [ ]:
print("\n" + "="*60)
print("Splitting Dataset")
print("="*60)

# Split: 80% train, 10% val, 10% test
train_val, test = train_test_split(rows, test_size=0.1, random_state=42)
train, val = train_test_split(train_val, test_size=0.111, random_state=42)  # 0.111 of 90% = 10% of total

print(f"Train set: {len(train)} ({len(train)/len(rows)*100:.1f}%)")
print(f"Val set: {len(val)} ({len(val)/len(rows)*100:.1f}%)")
print(f"Test set: {len(test)} ({len(test)/len(rows)*100:.1f}%)")

# Create output directory
out_dir = Path("../data/processed")
out_dir.mkdir(parents=True, exist_ok=True)

# Save as JSONL
print(f"\nSaving to {out_dir}...")
for name, data in [('train', train), ('val', val), ('test', test)]:
    filepath = out_dir / f'{name}.jsonl'
    with filepath.open('w', encoding='utf-8') as f:
        for row in data:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')
    print(f"  ✓ {name}.jsonl ({len(data)} samples)")

print(f"\n✓ Data preparation complete!")

In [ ]:
print("\n" + "="*60)
print("Sample Training Data")
print("="*60)

for i in range(min(2, len(train))):
    sample = train[i]
    print(f"\nExample {i+1}:")
    print(f"Instruction: {sample['instruction']}")
    print(f"Output: {sample['output'][:150]}..." if len(sample['output']) > 150 else f"Output: {sample['output']}")
    print(f"Context (preview): {sample['context'][:200]}..." if len(sample['context']) > 200 else f"Context: {sample['context']}")

## 3. Sample Data Verification

In [ ]:
print("Validating prepared data...")

for split_name, split_data in [('train', train), ('val', val), ('test', test)]:
    print(f"\n{split_name.upper()}:")
    
    # Calculate statistics
    instruction_lens = [len(x['instruction'].split()) for x in split_data]
    output_lens = [len(x['output'].split()) for x in split_data]
    context_lens = [len(x['context'].split()) for x in split_data]
    
    print(f"  Instruction length (words):")
    print(f"    min: {min(instruction_lens)}, max: {max(instruction_lens)}, avg: {np.mean(instruction_lens):.1f}")
    print(f"  Output length (words):")
    print(f"    min: {min(output_lens)}, max: {max(output_lens)}, avg: {np.mean(output_lens):.1f}")
    print(f"  Context length (words):")
    print(f"    min: {min(context_lens)}, max: {max(context_lens)}, avg: {np.mean(context_lens):.1f}")
    
    # Check for empty fields
    empty_counts = {
        'instruction': sum(1 for x in split_data if not x['instruction'].strip()),
        'output': sum(1 for x in split_data if not x['output'].strip()),
        'context': sum(1 for x in split_data if not x['context'].strip())
    }
    
    if any(empty_counts.values()):
        print(f"  ⚠ Empty fields: {empty_counts}")
    else:
        print(f"  ✓ No empty fields")

## 2. Data Quality Validation